# Outage-risk forecast: live demonstration

Weather-Driven Power-Outage Risk Forecasting.

This notebook runs the submitted system on a chosen period and shows the result. The predictions come only from the
official entry point `infer.py`, started as a separate process with the same arguments the committee uses; the notebook
checks the environment and the input, starts `infer.py`, checks its output and shows the summary. Everything runs
offline, and nothing in the package or the input folder is changed.

Run the cells from top to bottom (**Run All**).

## 1. Environment

Packages, device and the SHA-256 of every model file. The last line must be `ENVIRONMENT READY`.

In [ ]:
import sys
from pathlib import Path

PACKAGE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'infer.py').is_file())   # the package folder
sys.path.insert(0, str(PACKAGE / 'demo'))
import notebook_api as demo

demo.check_environment()

## 2. What to predict

Edit the options in the cell below and run it.

- `data`: the input folder (see the table). A relative path starts at the package folder.
- `start`, `end`: the issue times to predict, in `[start, end)`, UTC.
- `counties`: county names or FIPS codes.
- `tasks`: `'A'`, `'B'` or both.
- `generate_requests=True`: build the requests on the official grid. `False`: use the rows of the organizer request
  table that fall in the selection.

When the requests are generated, `start`, `end` and `counties` may be left out (`None`): the counties are then taken
from the MCC and the period from the outage records. More options (part of the lead times, Task B issue hours) are in
`docs/DEMO.md`.

### Input folder

The folder may have any name and be anywhere. The sub-folders and file names inside it are fixed; each file may be
`.csv` or `.parquet`.

| File | Content | Needed? |
| --- | --- | --- |
| `mcc/MCC.csv` | customers per county | **required** |
| `eaglei/eaglei_outages.csv` | outage records, every 5 or 15 minutes | expected; without it the history fallback is used |
| `weather/weather_forecasts.csv` | weather forecasts | expected; without it the weather-free model is used |
| `requests/forecast_requests.csv` | what to predict | optional here: generated on the official grid when absent |
| `radar/mrms_county_series.csv` | our radar series | optional |

Timestamps must carry a UTC mark (`Z` or an offset). The official command `python infer.py` itself always needs the
request table.

In [ ]:
options = dict(
    data='data/inputs/input_2025',
    start='2025-07-08',            # issue times in [start, end), UTC
    end='2025-07-15',
    counties=['Harris', 'Los Angeles', 'Miami-Dade', 'Wayne', 'Rockingham'],
    tasks=['A', 'B'],
    generate_requests=True,        # build the requests on the official grid; False = filter the organizer request table
    device='cpu',
)

## 3. Run

Five steps: environment, input, inference (`infer.py`), output check, reference score. The last line is `COMPLETED`
or `FAILED`; a failed run shows no predictions.

In [ ]:
run = demo.run(**options)
run

## 4. Reference score

Official formula, each task out of 100, all counties pooled. It is computed after the predictions were written,
against the observations in the input folder. A short period with few events gives an unstable score. The second
table is the breakdown by power architecture (diagnostic only).

In [ ]:
run.scores()

In [ ]:
run.scores(by='architecture')

## 5. Forecast and observation

One panel per county. Blue is the forecast share of customers without power, orange the observed share, and the grey
band the risk thresholds of the four architectures (above the band all four are at risk). The axis is linear near
zero and logarithmic above about 0.05%. `counties=` draws only some counties, larger.

In [ ]:
run.plot('A')

In [ ]:
run.plot('B')

In [ ]:
run.plot('B', counties=['Wayne'])

## 6. Which model produced the forecasts

The system chooses the path from the inputs that are present: full model, radar-free tree, weather-free tree, history
fallback.

In [ ]:
run.paths()

## 7. Checks and the output file

In [ ]:
run.checks()

In [ ]:
print(run.out / 'predictions.csv')
run.predictions().head(8)

## 8. Input ablation

The same requests without the radar table, then without weather and radar. The frozen system switches to its fallback
paths. These runs are stress tests, not the official output.

In [ ]:
no_radar = demo.run(**options, drop='radar')
no_weather_radar = demo.run(**options, drop=['weather', 'radar'])

In [ ]:
demo.compare(full=run, no_radar=no_radar, no_weather_radar=no_weather_radar)

In [ ]:
no_weather_radar.paths()

## 9. The official command

The run above is this command. Repeated in a terminal on the same device, it gives the same file.

In [ ]:
print('python', run.summary.get('command'))
print('SHA-256', run.summary.get('output', {}).get('sha256'))